Esse cogido é juntando o Play by play com a formação e o personel. Essa que tem dados de 2016 a 2024, a temporada de 2025 ainda não está disponivel (07/10/2026)

In [ ]:
import nflreadpy as nfl
import polars as pl



seasons = [2023, 2024]

pbp = nfl.load_pbp(seasons)
part = nfl.load_participation(seasons)

part_cols = part.select([
    "nflverse_game_id",
    "play_id",
    "offense_formation",
    "offense_personnel",
    "defense_personnel",
    "defenders_in_box",
    "number_of_pass_rushers",
]).with_columns(pl.col("play_id").cast(pl.Int64))

pbp = pbp.with_columns(pl.col("play_id").cast(pl.Int64))

pbp_full = pbp.join(
    part_cols,
    left_on=["game_id", "play_id"],
    right_on=["nflverse_game_id", "play_id"],
    how="left",
)
print(pbp_full.select([
    "offense_formation", "offense_personnel"
]).null_count())

print(pbp_full["offense_personnel"].drop_nulls().unique().head(10))

shape: (1, 2)
┌───────────────────┬───────────────────┐
│ offense_formation ┆ offense_personnel │
│ ---               ┆ ---               │
│ u32               ┆ u32               │
╞═══════════════════╪═══════════════════╡
│ 25516             ┆ 7070              │
└───────────────────┴───────────────────┘
shape: (10,)
Series: 'offense_personnel' [str]
[
	"1 C, 1 CB, 1 FB, 2 FS, 2 ILB, …
	"1 CB, 1 DE, 1 FS, 1 ILB, 1 OLB…
	"1 CB, 1 FB, 3 ILB, 1 LS, 1 P, …
	"1 CB, 1 FS, 3 ILB, 2 OLB, 2 RB…
	"2 CB, 1 FS, 3 ILB, 1 LS, 1 P, …
	"1 CB, 1 FS, 1 ILB, 1 MLB, 2 OL…
	"2 CB, 1 FS, 1 ILB, 1 OLB, 1 RB…
	"2 CB, 1 FS, 1 LS, 1 MLB, 3 OLB…
	"1 CB, 1 FB, 1 FS, 4 ILB, 1 LS,…
	"1 CB, 1 FB, 1 FS, 3 MLB, 1 OLB…
]


In [ ]:
import nflreadpy as nfl
import polars as pl

seasons = [2023, 2024]

part = nfl.load_participation(seasons)
pbp = nfl.load_pbp(seasons)

# Full participation data
part.write_csv("./CSVs/participation_2023_2024.csv")

# Just the columns you were looking at, plus the IDs
part.select([
    "nflverse_game_id",
    "play_id",
    "offense_formation",
    "offense_personnel",
]).write_csv("./CSVs/participation_formations.csv")

# pbp is large (hundreds of columns), so you may want a subset
pbp.select([
    "game_id", "play_id", "season", "week", "posteam", "defteam", "play_type",
]).write_csv("./CSVs/pbp_subset.csv")